# 技能0 · Day 6 上机：研究方法论入门 -- 营销 AI 领域文献计量

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **arxiv** 包查询真实 arXiv API，获取营销 AI 领域真实论文元数据
2. 用 **pandas** 做文献计量统计（按年份/作者/主题）
3. 用 **networkx** 构建作者合作网络与关键词共现网络
4. 用 **matplotlib** 可视化论文增长趋势与网络结构
5. 理解可复现研究（OSF 预注册 / FAIR 原则）为什么是 2026 年学术研究的基本要求

## 营销映射
本 Day 把"研究方法论"桥接到 AI + 企业营销：查询 arXiv "marketing analytics" / "causal inference marketing" / "LLM marketing" 主题论文，做文献计量分析，相当于营销技术选型前的"学术尽职调查"。


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 需要的库：arxiv（arXiv API 客户端）、pandas（文献计量）、networkx（网络分析）、matplotlib（可视化）。
> arXiv API 有速率限制（约 1 请求/3 秒），高频查询会收到 HTTP 429/503，此时自动切换到 fallback 样本。


In [ ]:
# !pip install arxiv pandas networkx matplotlib -q


## 1. 数据集背景与营销映射

**数据源**：arXiv API 实时查询（首选）或 `data/arxiv_fallback_sample.json`（fallback，18 篇真实论文元数据快照）。

| 查询主题 | arXiv query | 营销映射 | 研究方法论意义 |
|---------|------------|---------|-------------|
| marketing analytics | `"marketing analytics"` | 营销分析技术成熟度 | 文献综述第一步：系统检索 |
| causal inference marketing | `"causal inference marketing"` | 营销归因与增量建模基础 | 因果推断是营销归因的理论基础 |
| LLM marketing | `"LLM marketing"` | LLM 在营销中的应用前沿 | 新兴方向识别 |

**文献计量学（Bibliometrics）** 是研究方法论的核心方法之一：用统计方法分析学术文献的量化特征（论文数/引用数/合作网络/关键词共现），发现领域研究热度演化与新兴方向。


In [ ]:
import arxiv
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import os
from collections import Counter
from itertools import combinations

# Fallback data path
FALLBACK_PATH = os.path.join('data', 'arxiv_fallback_sample.json')

def load_fallback():
    """Load fallback arXiv sample when API is rate-limited (HTTP 429/503)."""
    with open(FALLBACK_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data['papers']

print("Libraries loaded.")
print("arxiv version:", arxiv.__version__)
print("pandas version:", pd.__version__)
print("networkx version:", nx.__version__)
print("Fallback path:", FALLBACK_PATH, "- exists:", os.path.exists(FALLBACK_PATH))


## TODO 1：用 arxiv 包查询 arXiv API（含 fallback）

**任务**：查询三个主题的 arXiv 论文，获取真实元数据。当 API 被速率限制（HTTP 429/503）或网络不通时，自动切换到 `data/arxiv_fallback_sample.json`。

**提示**：
- `arxiv.Client(num_retries=1, page_size=20)` 创建客户端
- `arxiv.Search(query=q, max_results=20, sort_by=arxiv.SortCriterion.Relevance)` 按关键词搜索
- 遍历 `client.results(search)`，提取 `r.entry_id`/`r.title`/`r.authors`/`r.published`/`r.summary`/`r.primary_category`
- `r.entry_id` 形如 `http://arxiv.org/abs/2210.03629v1`，用 `split('/')[-1]` 提取 arXiv ID
- 异常时用 `load_fallback()` 加载 fallback 样本

**理论连接**：文献综述的第一步是系统检索相关文献。arxiv 包让这一步可编程化。fallback 机制体现**可复现研究**原则--即使 API 不可用，分析也能复现。


In [ ]:
# TODO 1：用 arxiv 包查询 arXiv API，获取营销 AI 领域真实论文元数据
# 提示：用 arxiv.Search(query=q, max_results=20, sort_by=arxiv.SortCriterion.Relevance)
#       遍历 client.results(search)，提取 r.entry_id/r.title/r.authors/r.published/r.summary/r.primary_category
#       r.entry_id 形如 http://arxiv.org/abs/2210.03629v1，用 split('/')[-1] 提取 arXiv ID
#       异常时用 load_fallback() 加载 fallback 样本
# 要求：查询3个主题，返回论文列表 all_papers，打印论文总数和数据来源

QUERIES = ["marketing analytics", "causal inference marketing", "LLM marketing"]
all_papers = []
api_ok = False

# ===== 你的代码 =====
# TODO: 你的代码
raise NotImplementedError


## TODO 2：文献计量 -- 按年份统计论文增长趋势

**任务**：用 pandas 将论文列表转为 DataFrame，按年份统计论文数，观察营销 AI 领域的研究热度演化。

**提示**：
- `pd.DataFrame(all_papers)` 从列表创建 DataFrame
- `df['year'].value_counts().sort_index()` 按年份统计并排序
- 打印各年论文数，识别增长趋势

**数据治理视角**：年份字段应该是整数类型（int），检查是否有缺失值。论文增长趋势是判断一个技术领域"是否成熟"的关键信号--增长曲线越陡，说明学术界关注度越高。


In [ ]:
# TODO 2：文献计量 -- 按年份统计论文增长趋势
# 提示：用 pd.DataFrame(all_papers) 创建 DataFrame
#       用 df['year'].value_counts().sort_index() 按年份统计并排序
# 要求：创建 df，统计 year_counts，打印各年论文数

# ===== 你的代码 =====
# TODO: 你的代码
raise NotImplementedError


## TODO 3：高产作者排名与主题分布

**任务**：用 pandas 统计高产作者排名（Top 10）和按查询主题的论文分布。

**提示**：
- `df.groupby('query').size()` 按主题统计论文数
- `df.explode('authors')` 展开作者列表（一篇论文的多个作者各占一行）
- `authors_exploded['authors'].value_counts().head(10)` 统计高产作者 Top 10

**营销映射**：高产作者是领域的"意见领袖"，识别他们有助于追踪前沿研究方向。主题分布揭示三个查询主题的论文密度差异。


In [ ]:
# TODO 3：高产作者排名与主题分布
# 提示：用 df.groupby('query').size() 按主题统计论文数
#       用 df.explode('authors') 展开作者列表
#       用 authors_exploded['authors'].value_counts().head(10) 统计高产作者
# 要求：计算 topic_counts 和 author_counts，打印结果

# ===== 你的代码 =====
# TODO: 你的代码
raise NotImplementedError


## TODO 4：作者合作网络（networkx）

**任务**：用 networkx 构建作者合作网络（节点=作者，边=合作关系），计算度中心性识别核心作者。

**提示**：
- `nx.Graph()` 创建无向图
- 遍历每篇论文的作者列表，用 `itertools.combinations(authors, 2)` 生成所有作者对
- `G.add_edge(a, b, weight=1)` 添加合作边（已存在则 weight+1）
- `nx.degree_centrality(G)` 计算度中心性
- `sorted(deg_cent.items(), key=lambda x: -x[1])[:5]` 取 Top 5

**理论连接**：合作网络是文献计量学的核心方法。度中心性（Degree Centrality）= deg(v)/(n-1)，衡量作者的协作广度。高中心性作者是领域的"枢纽节点"。


In [ ]:
# TODO 4：作者合作网络（networkx）
# 提示：用 nx.Graph() 创建无向图
#       遍历每篇论文，用 combinations(authors, 2) 生成作者对
#       用 G.add_edge(a, b, weight=1) 添加边（已存在则 weight+1）
#       用 nx.degree_centrality(G) 计算度中心性
# 要求：构建 G_collab，打印节点数/边数和 Top 5 核心作者

G_collab = nx.Graph()

# ===== 你的代码 =====
# TODO: 你的代码
raise NotImplementedError


## TODO 5：关键词共现网络（networkx）

**任务**：用 networkx 构建关键词共现网络（节点=论文标题中的词，边=同一标题中共现），识别新兴研究方向。

**提示**：
- 对每篇论文标题分词（`title.lower().replace(':','').replace(',','').split()`）
- 过滤短词（`len(w) > 3`），用 `set()` 去重
- 用 `combinations(sorted(set(words)), 2)` 生成词对
- `G.add_edge(a, b, weight=1)` 添加共现边
- `sorted(G.degree, key=lambda x: -x[1])[:10]` 取 Top 10 关键词

**营销映射**：关键词共现网络揭示领域"概念地图"。高频共现词对是核心研究方向，低频新出现的词对可能是新兴方向。


In [ ]:
# TODO 5：关键词共现网络（networkx）
# 提示：对每篇论文标题分词：title.lower().replace(':','').replace(',','').split()
#       过滤短词：len(w) > 3，用 set() 去重
#       用 combinations(sorted(set(words)), 2) 生成词对
#       用 G.add_edge(a, b, weight=1) 添加共现边
# 要求：构建 G_kw，打印节点数/边数和 Top 10 关键词

G_kw = nx.Graph()

# ===== 你的代码 =====
# TODO: 你的代码
raise NotImplementedError


## TODO 6：可视化论文增长趋势与合作网络（matplotlib）

**任务**：用 matplotlib 绘制两个图：① 论文增长趋势柱状图 ② 作者合作网络图（最大连通子图）。

**提示**：
- `fig, axes = plt.subplots(1, 2, figsize=(14, 5))` 创建 1x2 子图
- `axes[0].bar(year_counts.index.astype(str), year_counts.values)` 柱状图
- `max(nx.connected_components(G), key=len)` 取最大连通子图
- `nx.spring_layout(sub, seed=42)` 力导向布局
- `nx.draw(sub, pos, ax=axes[1], node_size=30, with_labels=False)` 绘制网络
- `plt.savefig('output/day6_bibliometrics.png', dpi=100, bbox_inches='tight')` 保存

**可复现研究**：`seed=42` 固定随机种子，确保任何人复现都能得到相同的网络布局图。


In [ ]:
# TODO 6：可视化论文增长趋势与合作网络（matplotlib）
# 提示：用 fig, axes = plt.subplots(1, 2, figsize=(14, 5)) 创建 1x2 子图
#       用 axes[0].bar(...) 绘制年份柱状图
#       用 max(nx.connected_components(G_collab), key=len) 取最大连通子图
#       用 nx.spring_layout(sub, seed=42) 布局，nx.draw(sub, pos, ax=axes[1], ...) 绘制
# 要求：绘制双图，保存到 output/day6_bibliometrics.png

os.makedirs('output', exist_ok=True)

# ===== 你的代码 =====
# TODO: 你的代码
raise NotImplementedError


## 2. 反思与前沿

### 反思问题
1. 营销 AI 领域论文增长趋势如何？哪个年份论文最多？
2. 高产作者是谁？他们合作形成了怎样的网络结构？
3. 关键词共现网络揭示了哪些新兴方向？
4. 如果你要在营销领域做一个 A/B 测试研究，如何用 OSF 预注册？

### 2026 前沿：可复现研究 + ASReview + LLM 辅助文献综述

- **可复现研究**：`seed=42` 固定随机种子、`requirements.txt` 锁定依赖版本--确保任何人都能复现你的文献计量分析
- **OSF 预注册**：在数据收集前公开注册研究假设，对抗 p-hacking 与发表偏倚
- **FAIR 原则**：数据与代码应 Findable/Accessible/Interoperable/Reusable
- **ASReview**（`pip install asreview`）：AI 辅助系统性文献综述，用主动学习筛选论文，比人工快 10x
- **LLM 辅助研究**：用 DeepSeek 等开源 LLM 做文献摘要提取，但必须用 arxiv 包验证论文真实存在（防止 LLM 幻觉）

> ⚠️ LLM 辅助文献综述是加速工具，不是替代工具。LLM 可能幻觉出不存在的论文--这就是为什么本 Day 用 arxiv 包查询**真实 API 返回的真实论文元数据**。
